# 尝试基于smp使用哨兵数据进行建筑识别

In [ ]:
import torch
import segmentation_models_pytorch as smp
import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal

def build_sentinel_model():
    # 1. 配置设备 (适配你的 5060 Ti)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 2. 定义 SMP 模型
    # 针对哨兵2号 RGB 数据：in_channels=3
    # 建筑物提取通常为二分类：classes=1 (使用 Sigmoid 激活)
    model = smp.Unet(
        encoder_name="resnet34",        # ResNet34 轻量且收敛快，适合初学者
        encoder_weights="imagenet",     # 使用预训练权重
        in_channels=3,                  
        classes=1,                      
        activation='sigmoid'            
    ).to(device)

    return model, device

def preprocess_sentinel(image_rgb):
    """
    针对哨兵2号数据的预处理：
    哨兵数据通常是 uint16 (0-10000)，需要拉伸到 0-1 并转为 Float32
    """
    # 将 uint16 转为 float32 并归一化到 [0, 1]
    image = image_rgb.astype(np.float32) / image_rgb.max()  
    image = np.clip(image, 0, 1)
    
    # 调整维度从 [H, W, C] 到 [C, H, W] 并增加 Batch 维度
    img = torch.from_numpy(image).permute(2, 0, 1).unsqueeze(0)

    # 2-98% 拉伸（逐通道）
    for i in range(3):
        p2, p98 = np.percentile(img[:, :, i], (2, 98))
        img[:, :, i] = np.clip((img[:, :, i] - p2) / (p98 - p2), 0, 1)

    return img

# --- 执行模拟测试 ---
model, device = build_sentinel_model()
model.eval()

gdal.UseExceptions()
data_path=("/mnt/d/pyLearn/pyLearn/test_project/data/tif/"
        "S2B_MSIL2A_20230611T030529_N0509_R075_T49RFL_20230611T053545_10m_7bands.tif")
with gdal.Open(data_path) as src:
     band=src.ReadAsArray()

rgb=band[[2,1,0],:,:] # 选择需要的波段，注意波段索引可能需要调整
input_tensor = preprocess_sentinel(rgb).to(device)

with torch.no_grad():
    # 使用混合精度推理，加速 5060 Ti 计算
    with torch.cuda.amp.autocast():
        prediction = model(input_tensor)
        # 将结果转回二值掩码 (阈值 0.5)
        mask = (prediction > 0.5).cpu().numpy().squeeze()

print(f"分割完成！Mask 尺寸: {mask.shape}")

In [4]:
from osgeo import gdal
gdal.UseExceptions()
data_path=("/mnt/d/pyLearn/pyLearn/test_project/data/tif/"
        "S2B_MSIL2A_20230611T030529_N0509_R075_T49RFL_20230611T053545_10m_7bands.tif")
with gdal.Open(data_path) as src:
     band=src.ReadAsArray()
     nodata = src.GetRasterBand(1).GetNoDataValue()
     print(f"NoData Value: {nodata}")

NoData Value: None
